In [6]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("train.csv")

# Kiểm tra nhanh số dòng, số cột
print(f"Dataset shape: {df.shape}")
df[["Id", "SalePrice", "GrLivArea", "OverallQual", "Neighborhood"]].head()

Dataset shape: (1460, 81)


,Id,SalePrice,GrLivArea,OverallQual,Neighborhood
0,1,208500,1710,7,CollgCr
1,2,181500,1262,6,Veenker
2,3,223500,1786,7,CollgCr
3,4,140000,1717,7,Crawfor
4,5,250000,2198,8,NoRidge


In [7]:
# --- PHƯƠNG PHÁP 1: IQR ---
q1 = df["SalePrice"].quantile(0.25)
q3 = df["SalePrice"].quantile(0.75)
iqr = q3 - q1

lower_iqr = q1 - 1.5 * iqr
upper_iqr = q3 + 1.5 * iqr

outliers_iqr = df[
    (df["SalePrice"] < lower_iqr) | (df["SalePrice"] > upper_iqr)
]

# --- PHƯƠNG PHÁP 2: Z-SCORE ---
mean_sp = df["SalePrice"].mean()
std_sp = df["SalePrice"].std()

df["SalePrice_Zscore"] = (df["SalePrice"] - mean_sp) / std_sp
outliers_zscore = df[df["SalePrice_Zscore"].abs() > 3]

# --- SO SÁNH KẾT QUẢ ---
print(
    f"IQR: {len(outliers_iqr)} outliers (Ngưỡng: [{lower_iqr:,.1f}, {upper_iqr:,.1f}])"
)
print(
    f"Z-score (|z| > 3): {len(outliers_zscore)} outliers (Ngưỡng: > {mean_sp + 3*std_sp:,.1f})"
)

common = set(outliers_iqr["Id"]).intersection(set(outliers_zscore["Id"]))
print(f"Số lượng outliers trùng nhau ở cả 2 phương pháp: {len(common)}")

IQR: 61 outliers (Ngưỡng: [3,937.5, 340,037.5])
Z-score (|z| > 3): 22 outliers (Ngưỡng: > 419,248.7)
Số lượng outliers trùng nhau ở cả 2 phương pháp: 22


In [8]:
# Lấy 5 outlier có SalePrice cao nhất (độ lệch dương lớn nhất)
cols_investigate = [
    "Id",
    "SalePrice",
    "SalePrice_Zscore",
    "Neighborhood",
    "OverallQual",
    "OverallCond",
    "GrLivArea",
    "TotalBsmtSF",
    "YearBuilt",
    "GarageCars",
    "SaleType",
    "SaleCondition",
]

top5_outliers = (
    df.sort_values(by="SalePrice", ascending=False)
    .head(5)[cols_investigate]
    .copy()
)
top5_outliers

,Id,SalePrice,SalePrice_Zscore,Neighborhood,OverallQual,OverallCond,GrLivArea,TotalBsmtSF,YearBuilt,GarageCars,SaleType,SaleCondition
691,692,755000,7.226343,NoRidge,10,6,4316,2444,1994,3,WD,Normal
1182,1183,745000,7.100466,NoRidge,10,5,4476,2396,1996,3,WD,Abnorml
1169,1170,625000,5.589940,NoRidge,10,5,3627,1930,1995,3,WD,Normal
898,899,611657,5.421982,NridgHt,9,5,2364,2330,2009,3,New,Partial
803,804,582933,5.060412,NridgHt,9,5,2822,1734,2008,3,New,Partial


In [9]:
# --- 1. CỘT Exterior2nd ---
ext2_before = df["Exterior2nd"].nunique()

df["Exterior2nd"] = df["Exterior2nd"].str.strip()
map_ext2 = {
    "Wd Shng": "WdShing",  # Đồng bộ không dấu cách với data_description.txt
    "Brk Cmn": "BrkComm",  # Đồng bộ mã với Exterior1st
    "CmentBd": "CemntBd",  # Sửa lỗi chính tả
}
df["Exterior2nd"] = df["Exterior2nd"].replace(map_ext2)
ext2_after = df["Exterior2nd"].nunique()

print(f"Column: Exterior2nd")
print(f"Before: {ext2_before} distinct values")
print(f"After: {ext2_after} distinct values\n")

# --- 2. CỘT MSZoning ---
zoning_before = df["MSZoning"].nunique()

df["MSZoning"] = df["MSZoning"].str.strip()
df["MSZoning"] = df["MSZoning"].replace({"C (all)": "C"})
zoning_after = df["MSZoning"].nunique()

print(f"Column: MSZoning")
print(f"Before: {zoning_before} distinct values")
print(f"After: {zoning_after} distinct values")

Column: Exterior2nd
Before: 16 distinct values
After: 16 distinct values

Column: MSZoning
Before: 5 distinct values
After: 5 distinct values


In [10]:
# Tạo bảng Cleaning Log ghi nhận toàn bộ thao tác
cleaning_log = pd.DataFrame(
    [
        {
            "Step ID": "LOG-01",
            "Target Column": "SalePrice",
            "Issue Identified": "Phân phối lệch phải, chứa nhiều giá trị cực lớn",
            "Action Taken": "Tính ngưỡng IQR và Z-score (|Z| > 3)",
            "Rationale": "Phát hiện outlier bằng phương pháp tham số và phi tham số",
            "Impact": "IQR gắn cờ 61 dòng; Z-score gắn cờ 22 dòng",
        },
        {
            "Step ID": "LOG-02",
            "Target Column": "SalePrice & đặc tính nhà",
            "Issue Identified": "5 căn nhà có giá từ 582,933 đến 755,000 USD",
            "Action Taken": "Khảo sát tính hợp lý; kết luận Real và giữ lại (Retain)",
            "Rationale": "Biệt thự cao cấp thực tế, không phải lỗi nhập liệu",
            "Impact": "Không xóa dòng nào (giữ nguyên 1,460 dòng)",
        },
        {
            "Step ID": "LOG-03",
            "Target Column": "Exterior2nd",
            "Issue Identified": "Lỗi khoảng trắng và sai chính tả ('Wd Shng', 'Brk Cmn', 'CmentBd')",
            "Action Taken": "Dùng .str.strip() và .replace() để map về mã chuẩn",
            "Rationale": "Đồng bộ chuẩn theo data_description.txt và Exterior1st",
            "Impact": "105 dòng được chuẩn hóa",
        },
        {
            "Step ID": "LOG-04",
            "Target Column": "MSZoning",
            "Issue Identified": "Mã khu thương mại ghi dạng diễn giải 'C (all)'",
            "Action Taken": "Dùng .str.strip() và .replace({'C (all)': 'C'})",
            "Rationale": "Chuẩn hóa theo ký hiệu phân vùng 1-2 chữ cái",
            "Impact": "10 dòng được chuẩn hóa",
        },
    ]
)

cleaning_log

,Step ID,Target Column,Issue Identified,Action Taken,Rationale,Impact
0,LOG-01,SalePrice,"Phân phối lệch phải, chứa nhiều giá trị cực lớn",Tính ngưỡng IQR và Z-score (|Z| > 3),Phát hiện outlier bằng phương pháp tham số và ...,IQR gắn cờ 61 dòng; Z-score gắn cờ 22 dòng
1,LOG-02,SalePrice & đặc tính nhà,"5 căn nhà có giá từ 582,933 đến 755,000 USD",Khảo sát tính hợp lý; kết luận Real và giữ lại...,"Biệt thự cao cấp thực tế, không phải lỗi nhập ...","Không xóa dòng nào (giữ nguyên 1,460 dòng)"
2,LOG-03,Exterior2nd,"Lỗi khoảng trắng và sai chính tả ('Wd Shng', '...",Dùng .str.strip() và .replace() để map về mã c...,Đồng bộ chuẩn theo data_description.txt và Ext...,105 dòng được chuẩn hóa
3,LOG-04,MSZoning,Mã khu thương mại ghi dạng diễn giải 'C (all)',Dùng .str.strip() và .replace({'C (all)': 'C'}),Chuẩn hóa theo ký hiệu phân vùng 1-2 chữ cái,10 dòng được chuẩn hóa
